# Práctica 1: Segmentación de imágenes con K-means

**Imagen:** `tlayoyos.jpg`  
**Objetivo:** agrupar píxeles con colores similares y observar qué ocurre al reducir el número de colores disponibles.

## 1. Importar librerías y cargar la imagen

In [ ]:
filename = "que-es-un-tlacoyo-6.jpg"

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

image = Image.open(filename).convert("RGB")
image = np.asarray(image)
print(image.shape)

In [ ]:
plt.imshow(image)
plt.title(filename) 
plt.axis("off")
plt.show()

## 2. Representar los píxeles

Una imagen RGB tiene dimensiones `(alto, ancho, 3)`. Cada píxel se representa como un vector `(R, G, B)`, así que convertimos la imagen en una matriz donde cada fila es un píxel.

In [ ]:
X = image.reshape(-1, 3)
print(f"Matriz X: {X.shape}")

## 3. Segmentar usando K-means

Se prueban los valores `K = 2, 3, 4, 5, 6, 7, 8, 10`. Cada centroide representa un color prototipo y cada píxel se reemplaza por el centroide de su cluster.

In [ ]:
from sklearn.cluster import KMeans

k = 8

kmeans = KMeans(
    n_clusters=k,
    random_state=42
    )

kmeans.fit(X)

In [ ]:
segmented_img = \
kmeans.cluster_centers_[kmeans.labels_]
segmented_img = segmented_img.reshape(image.shape)
segmented_img = segmented_img.astype(np.uint8)

plt.imshow(segmented_img)
plt.axis("off")
plt.title(filename + " segmentada con " + str(k) + " clusters")
plt.show()

In [ ]:
k_values = [2, 3, 4, 5, 6, 7, 8, 10]
segmented_images = {}
models = {}

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X)
    segmented_img = kmeans.cluster_centers_[kmeans.labels_]
    segmented_img = segmented_img.reshape(image.shape).astype(np.uint8)
    models[k] = kmeans
    segmented_images[k] = segmented_img


fig, axes = plt.subplots(3, 3, figsize=(14, 13))

axes = axes.ravel()
axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis("off")

for ax, k in zip(axes[1:], k_values):
    ax.imshow(segmented_images[k])
    ax.set_title(f"K={k}")
    ax.axis("off")

plt.suptitle("Segmentación por color con K-means", fontsize=16)
plt.tight_layout()
plt.show()

## Preguntas

1. **¿Qué representa un cluster en este problema?**  
   Un cluster representa un grupo de píxeles cuyos colores RGB son parecidos.

2. **¿Qué representa cada centroide?**  
   Cada centroide representa el color RGB promedio o prototipo de un cluster y es el color que se asigna a los píxeles de ese grupo.

3. **¿Qué ocurre cuando K es muy pequeño?**  
   Hay pocos colores y se ve muy simplificada.

4. **¿Qué ocurre al aumentar K?**  
   Aparecen más detalles, pero también aumenta la complejidad y la mejora visual llega a ser cada vez menos notable.

5. **¿Qué valor de K considero más adecuado para esta imagen?**  
   Depende. Si quisiera clasificar entre tlacoyos verdes y rojos debería hacer antes una transformación que se centre mas en los tlacoyos, como recortar la imagen quitando los bordes, porque (creo) que como hay mas fondo de color naranja, azul, y amarillo, le da mas "peso" o "prioridad" a encontrar centroides ahí. Si quisiera contar tlacoyos creo que con K=3 podría funcionar.

6. **¿K-means está identificando objetos o únicamente colores?**  
   Como está agrupando puntos en el espacio RGB (creo que) identifica colores. No conoce la forma, la posición ni el significado de los objetos, si no no colorearía el mantel del fondo.
   Supongo que depende del problema: si tus objetos son de colores específicos (A <>=> azul, B <=> rojo, C <>=> amarillo, ...) y las imágenes son "buenas", entonces: si reconoce colores, entonces reconoce objetos, 

Abajo agregué visualizaciones bonitas

# Visualizaciones

Cada punto representa un píxel. Sus coordenadas son `(R, G, B)` y el color del punto es el RGB real del píxel. Se toma una muestra aleatoria para que la figura sea fluida al interactuar con ella.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# CONFIGURACIÓN
# ============================================================

rng = np.random.default_rng(42)

sample_size = min(15000, len(X))
sample_idx = rng.choice(len(X), size=sample_size, replace=False)

rgb_sample = X[sample_idx]

# Colores RGB reales de cada píxel
rgb_colors = [
    f"rgb({int(r)},{int(g)},{int(b)})"
    for r, g, b in rgb_sample
]

# Valores de K disponibles
k_values = sorted(models.keys())

# ============================================================
# FIGURA: DOS VENTANAS 3D HORIZONTALES
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[
        [{"type": "scene"}, {"type": "scene"}]
    ],
    subplot_titles=(
        "Distribución RGB original",
        "RGB coloreado por clusters"
    ),
    horizontal_spacing=0.04
)

# ============================================================
# CREAR TRACES PARA CADA K
# ============================================================

# Guardaremos qué traces pertenecen a cada K.
#
# Por cada K agregamos 4 traces:
#
# 0. Píxeles RGB originales
# 1. Centroides sobre RGB original
# 2. Píxeles coloreados por cluster
# 3. Centroides sobre clustering
#
# Solo los del primer K estarán visibles inicialmente.
# ============================================================

for k_index, k in enumerate(k_values):

    model = models[k]

    centers = model.cluster_centers_

    labels_sample = model.predict(rgb_sample)

    center_colors = [
        f"rgb({r:.0f},{g:.0f},{b:.0f})"
        for r, g, b in centers
    ]

    point_centroid_colors = [
        center_colors[label]
        for label in labels_sample
    ]

    visible = (k_index == 0)

    # --------------------------------------------------------
    # IZQUIERDA: píxeles RGB originales
    # --------------------------------------------------------

    fig.add_trace(
        go.Scatter3d(
            x=rgb_sample[:, 0],
            y=rgb_sample[:, 1],
            z=rgb_sample[:, 2],

            mode="markers",

            name="Píxeles RGB",

            marker=dict(
                size=3,
                color=rgb_colors,
                opacity=0.55
            ),

            text=[
                f"R={r}, G={g}, B={b}"
                for r, g, b in rgb_sample
            ],

            hovertemplate="%{text}<extra></extra>",

            visible=visible,

            showlegend=False
        ),

        row=1,
        col=1
    )

    # --------------------------------------------------------
    # IZQUIERDA: centroides
    # --------------------------------------------------------

    fig.add_trace(
        go.Scatter3d(
            x=centers[:, 0],
            y=centers[:, 1],
            z=centers[:, 2],

            mode="markers+text",

            name=f"Centroides K={k}",

            text=[
                f"C{i + 1}"
                for i in range(k)
            ],

            textposition="top center",

            marker=dict(
                size=10,
                color=center_colors,
                line=dict(
                    width=2,
                    color="black"
                )
            ),

            hovertemplate=[
                (
                    f"Centroide C{i + 1}<br>"
                    f"R={r:.1f}<br>"
                    f"G={g:.1f}<br>"
                    f"B={b:.1f}"
                    "<extra></extra>"
                )
                for i, (r, g, b) in enumerate(centers)
            ],

            visible=visible,

            showlegend=False
        ),

        row=1,
        col=1
    )

    # --------------------------------------------------------
    # DERECHA: píxeles coloreados por su cluster
    # --------------------------------------------------------

    fig.add_trace(
        go.Scatter3d(
            x=rgb_sample[:, 0],
            y=rgb_sample[:, 1],
            z=rgb_sample[:, 2],

            mode="markers",

            name="Clusters",

            marker=dict(
                size=3,
                color=point_centroid_colors,
                opacity=0.35
            ),

            text=[
                (
                    f"Píxel: R={r}, G={g}, B={b}"
                    f"<br>Cluster={label + 1}"
                )
                for (r, g, b), label
                in zip(rgb_sample, labels_sample)
            ],

            hovertemplate="%{text}<extra></extra>",

            visible=visible,

            showlegend=False
        ),

        row=1,
        col=2
    )

    # --------------------------------------------------------
    # DERECHA: centroides
    # --------------------------------------------------------

    fig.add_trace(
        go.Scatter3d(
            x=centers[:, 0],
            y=centers[:, 1],
            z=centers[:, 2],

            mode="markers+text",

            name=f"Centroides K={k}",

            text=[
                f"C{i + 1}"
                for i in range(k)
            ],

            textposition="top center",

            marker=dict(
                size=8,
                color=center_colors,
                line=dict(
                    width=2,
                    color="black"
                )
            ),

            hovertemplate=[
                (
                    f"Centroide C{i + 1}<br>"
                    f"R={r:.1f}<br>"
                    f"G={g:.1f}<br>"
                    f"B={b:.1f}"
                    "<extra></extra>"
                )
                for i, (r, g, b) in enumerate(centers)
            ],

            visible=visible,

            showlegend=False
        ),

        row=1,
        col=2
    )


# ============================================================
# BOTONES PARA CAMBIAR K
# ============================================================

buttons = []

traces_per_k = 4
total_traces = len(k_values) * traces_per_k

for k_index, k in enumerate(k_values):

    visibility = [False] * total_traces

    start = k_index * traces_per_k

    visibility[start:start + traces_per_k] = [True] * traces_per_k

    buttons.append(
        dict(
            label=f"K = {k}",

            method="update",

            args=[
                {
                    "visible": visibility
                },
                {
                    "title": (
                        f"K-Means sobre espacio RGB — "
                        f"K={k} — "
                        f"{sample_size:,} píxeles"
                    )
                }
            ]
        )
    )


# ============================================================
# CONFIGURAR EJES 3D
# ============================================================

scene_config = dict(

    xaxis=dict(
        title="R (rojo)",
        range=[0, 255]
    ),

    yaxis=dict(
        title="G (verde)",
        range=[0, 255]
    ),

    zaxis=dict(
        title="B (azul)",
        range=[0, 255]
    ),

    aspectmode="cube"
)


# ============================================================
# LAYOUT FINAL
# ============================================================

fig.update_layout(

    title=(
        f"K-Means sobre espacio RGB — "
        f"K={k_values[0]} — "
        f"{sample_size:,} píxeles"
    ),

    template="plotly_white",

    scene=scene_config,
    scene2=scene_config,

    width=1500,
    height=750,

    margin=dict(
        l=0,
        r=0,
        t=110,
        b=0
    ),

    updatemenus=[
        dict(
            type="dropdown",

            buttons=buttons,

            direction="down",

            x=0.5,
            y=1.12,

            xanchor="center",
            yanchor="top",

            showactive=True
        )
    ]
)

fig.show()

## Dashboard

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 1) CONFIGURACIÓN
# ============================================================

# K's que quieres mostrar
k_values = [2, 3, 4, 5, 6, 7, 8, 10]

# Para que no pese demasiado el dashboard:
# 9 escenas 3D con 15k puntos cada una puede ser muy pesado.
rng = np.random.default_rng(42)
sample_size = min(4000, len(X))   # puedes subirlo si tu compu aguanta
sample_idx = rng.choice(len(X), size=sample_size, replace=False)
rgb_sample = X[sample_idx]

# Colores RGB reales de la muestra
rgb_colors = [
    f"rgb({int(r)},{int(g)},{int(b)})"
    for r, g, b in rgb_sample
]

# ============================================================
# 2) SUBPLOTS 9x2
#    Fila 1: original
#    Filas 2..9: una por cada k
# ============================================================

n_rows = 1 + len(k_values)  # 1 fila original + 8 de los k's = 9

specs = [[{"type": "xy"}, {"type": "scene"}] for _ in range(n_rows)]

subplot_titles = [
    "Imagen original", "Distribución RGB original"
]

for k in k_values:
    subplot_titles.extend([
        f"Segmentación K={k}",
        f"RGB coloreado por clusters (K={k})"
    ])

fig = make_subplots(
    rows=n_rows,
    cols=2,
    specs=specs,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.04,
    vertical_spacing=0.03
)

# ============================================================
# 3) FILA 1
#    Izquierda: imagen original
#    Derecha: nube RGB original
# ============================================================

# Izquierda: imagen original
fig.add_trace(
    go.Image(z=image),
    row=1,
    col=1
)

# Derecha: RGB original en R^3
fig.add_trace(
    go.Scatter3d(
        x=rgb_sample[:, 0],
        y=rgb_sample[:, 1],
        z=rgb_sample[:, 2],
        mode="markers",
        name="RGB original",
        marker=dict(
            size=2.5,
            color=rgb_colors,
            opacity=0.6
        ),
        text=[f"R={r}, G={g}, B={b}" for r, g, b in rgb_sample],
        hovertemplate="%{text}<extra></extra>",
        showlegend=False
    ),
    row=1,
    col=2
)

# ============================================================
# 4) FILAS 2..9
#    Izquierda: imagen segmentada
#    Derecha: nube RGB coloreada por cluster + centroides
# ============================================================

for i, k in enumerate(k_values, start=2):

    model = models[k]
    centers = model.cluster_centers_

    # Predicción sobre la misma muestra usada en los 3D
    labels_sample = model.predict(rgb_sample)

    # Colores de centroides
    center_colors = [
        f"rgb({int(round(r))},{int(round(g))},{int(round(b))})"
        for r, g, b in centers
    ]

    # Cada punto toma el color de su centroide
    point_centroid_colors = [center_colors[label] for label in labels_sample]

    # --------------------------------------------------------
    # Columna izquierda: imagen segmentada
    # --------------------------------------------------------
    fig.add_trace(
        go.Image(z=segmented_images[k]),
        row=i,
        col=1
    )

    # --------------------------------------------------------
    # Columna derecha: nube RGB clusterizada
    # --------------------------------------------------------
    fig.add_trace(
        go.Scatter3d(
            x=rgb_sample[:, 0],
            y=rgb_sample[:, 1],
            z=rgb_sample[:, 2],
            mode="markers",
            name=f"Píxeles K={k}",
            marker=dict(
                size=2.5,
                color=point_centroid_colors,
                opacity=0.35
            ),
            text=[
                f"R={r}, G={g}, B={b} | Cluster={label+1}"
                for (r, g, b), label in zip(rgb_sample, labels_sample)
            ],
            hovertemplate="%{text}<extra></extra>",
            showlegend=False
        ),
        row=i,
        col=2
    )

    # Centroides
    fig.add_trace(
        go.Scatter3d(
            x=centers[:, 0],
            y=centers[:, 1],
            z=centers[:, 2],
            mode="markers+text",
            name=f"Centroides K={k}",
            text=[f"C{j+1}" for j in range(k)],
            textposition="top center",
            marker=dict(
                size=7,
                color=center_colors,
                line=dict(width=2, color="black")
            ),
            hovertemplate=[
                f"Centroide C{j+1}: R={r:.1f}, G={g:.1f}, B={b:.1f}<extra></extra>"
                for j, (r, g, b) in enumerate(centers)
            ],
            showlegend=False
        ),
        row=i,
        col=2
    )

# ============================================================
# 5) AJUSTES DE EJES
# ============================================================

# Ocultar ejes en todas las imágenes de la izquierda
for r in range(1, n_rows + 1):
    fig.update_xaxes(visible=False, row=r, col=1)
    fig.update_yaxes(visible=False, row=r, col=1)

# Configuración de todas las escenas 3D
scene_style = dict(
    xaxis=dict(title="R", range=[0, 255]),
    yaxis=dict(title="G", range=[0, 255]),
    zaxis=dict(title="B", range=[0, 255]),
    aspectmode="cube"
)

# Los nombres internos de escenas son: scene, scene2, scene3, ...
for s in range(1, n_rows + 1):
    scene_name = "scene" if s == 1 else f"scene{s}"
    fig.layout[scene_name].update(scene_style)

# ============================================================
# 6) LAYOUT FINAL
# ============================================================

fig.update_layout(
    title=f"Dashboard de segmentación por color con K-means ({sample_size:,} píxeles muestreados para los gráficos 3D)",
    template="plotly_white",
    width=1500,
    height=320 * n_rows,  # ajusta si quieres más/menos altura
    margin=dict(l=20, r=20, t=80, b=20)
)

fig.show()
